# Discogs Collector (Snowflake Notebook)
This notebook runs inside Snowflake and ingests releases + artwork into DISCOGS_DB.COLLECTION_DATA.

Prereqs:
- External Access Integration (EAI_DISCOGS) to `api.discogs.com:443`
- Secret `DISCOGS_API_TOKEN` stored in the same database/schema
- Tables created as per the schema guide (ARTWORK.IMAGE as FILE)


In [ ]:
import requests, json, io
from snowflake.snowpark.context import get_active_session
import _snowflake

session = get_active_session()
DATABASE = 'DISCOGS_DB'
SCHEMA = 'COLLECTION_DATA'
QUAL = f'{DATABASE}.{SCHEMA}'

# read token from secret
token = _snowflake.get_generic_secret_string(f'{QUAL}.DISCOGS_API_TOKEN')
headers = {
    'User-Agent': 'DiscogsCollector/1.0',
    'Authorization': f'Discogs token={token}'
}


## Helper: Insert/Update Release and Artwork


In [ ]:
def insert_artwork_file(artwork_id: str, release_id: str, image_type: str, url: str, content_bytes: bytes, fmt: str = 'jpeg'):
    session.sql(f"""
      INSERT INTO {QUAL}.ARTWORK (ARTWORK_ID, RELEASE_ID, IMAGE_TYPE, ORIGINAL_URL, IMAGE, FILE_SIZE, FILE_FORMAT, DOWNLOAD_DATE)
      SELECT '{artwork_id}', '{release_id}', '{image_type}', '{url}', TO_FILE(%s, '{fmt}'), {len(content_bytes)}, '{fmt}', CURRENT_TIMESTAMP()
    """, params=[content_bytes]).collect()

# Extend with your own upsert-release logic following your schema



## Example: Fetch a release and store artwork


In [ ]:
release_id = 249504  # example
r = requests.get(f'https://api.discogs.com/releases/{release_id}', headers=headers)
r.raise_for_status()
release = r.json()

# download images to FILE column
images = release.get('images', []) or []
for i, img in enumerate(images):
    url = img.get('uri')
    if not url:
        continue
    ir = requests.get(url, timeout=30)
    ir.raise_for_status()
    img_bytes = ir.content
    art_id = f'artwork_{release_id}_{i}'
    insert_artwork_file(art_id, str(release_id), img.get('type','image'), url, img_bytes, fmt='jpeg')

print('Stored artwork FILEs for release', release_id)
